# CompVP Part 1: Marigold normals (Result2) and Qwen3-VL (Result4) on Colab
Pretrained models only, no fine-tuning. Runtime must be **T4 GPU** (Runtime → Change runtime type).
Run the cells top to bottom. Upload `colab_upload.zip` to the top level of **My Drive** first.

In [ ]:
# Step 1: check that a GPU is attached (should say "Tesla T4"). If you see "command not found", the runtime has no GPU.
!nvidia-smi

In [ ]:
# Step 2: mount Google Drive (a popup asks you to allow access)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Step 3: unzip the project data into /content/project and count the files
!unzip -q -o /content/drive/MyDrive/colab_upload.zip -d /content/project
!for d in vision Original Mooney GT/depth GT/normal; do echo "$d: $(ls /content/project/$d | wc -l) files"; done
# Expected: vision 6, Original 100, Mooney 101 (100 PNG + log), GT/depth 100, GT/normal 100

In [ ]:
# Step 4: install the model libraries. Colab's own torch is kept (it is not in this list).
# -U because Qwen3-VL needs a recent transformers (local versions: transformers 5.22, diffusers 0.42).
!pip install -q -U transformers diffusers accelerate
import torch, transformers, diffusers
print("torch", torch.__version__, "| transformers", transformers.__version__, "| diffusers", diffusers.__version__,
      "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")

In [ ]:
# Step 5: Marigold smoke test, 2 originals + 2 Mooneys, in fp16 (half precision).
# If step 6 shows NaNs, set NORMALS_FLAGS = "--fp32" and rerun this cell.
NORMALS_FLAGS = ""
%cd /content/project/vision
!python run_normals.py --limit 2 {NORMALS_FLAGS}

In [ ]:
# Step 6: show the smoke-test normal maps and check the raw .npy files for NaN (should be 0.0)
import numpy as np, matplotlib.pyplot as plt
from pathlib import Path
R = Path('/content/project/Result2')
pngs = sorted(R.glob('*/*.png'))
fig, ax = plt.subplots(1, len(pngs), figsize=(4 * len(pngs), 4))
for a, p in zip(np.atleast_1d(ax), pngs):
    a.imshow(plt.imread(p)); a.set_title(f'{p.parent.name}/{p.stem}', fontsize=9); a.axis('off')
plt.show()
for f in sorted(R.glob('*/*.npy')):
    x = np.load(f); print(f'{f.parent.name}/{f.name}: NaN fraction {np.isnan(x).mean():.3f}, std {np.nanstd(x):.3f}')
print(open(R / 'metrics.csv').read())

In [ ]:
# Step 7: full Marigold run, all 100 originals + 100 Mooneys (overwrites the smoke-test files).
# Uses the same NORMALS_FLAGS as step 5. Prints one line per image.
# The last line also backs Result2/ up to My Drive (compvp_backup/) so a later disconnect doesn't lose it.
import time; t = time.time()
%cd /content/project/vision
!python run_normals.py {NORMALS_FLAGS}
print(f'Total: {(time.time() - t) / 60:.1f} min')
!python -c "import numpy as np, glob; f=glob.glob('../Result2/*/*.npy'); print(len(f), 'files, any NaN:', any(np.isnan(np.load(x)).any() for x in f))"
!mkdir -p /content/drive/MyDrive/compvp_backup && cp -r /content/project/Result2 /content/drive/MyDrive/compvp_backup/ && echo 'Result2 backed up to My Drive'

In [ ]:
# Step 8: check that the GPU is empty again before Qwen loads (each script runs as its own process,
# so Marigold has already freed its memory; expect only a few hundred MiB used)
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

In [ ]:
# Step 9: Qwen3-VL smoke test, 2 originals + 2 Mooneys, then print the raw answers.
# fp32 on purpose: the T4 has no native bf16, and the 2B model in fp32 (~8.5 GB) fits in its 16 GB.
VLM_FLAGS = "--fp32"
from pathlib import Path
%cd /content/project/vision
!python run_vlm.py --limit 2 {VLM_FLAGS}
for f in sorted(Path('/content/project/Result4').glob('*/*.txt')):
    print(f'--- {f.parent.name}/{f.name}\n{f.read_text()}')

In [ ]:
# Step 10: full Qwen run, all 200 images (overwrites the smoke-test files)
import time; t = time.time()
%cd /content/project/vision
!python run_vlm.py {VLM_FLAGS}
print(f'Total: {(time.time() - t) / 60:.1f} min')
import pandas as pd
m = pd.read_csv('/content/project/Result4/metrics.csv')
print(m.groupby('split').auto_correct.mean())
!mkdir -p /content/drive/MyDrive/compvp_backup && cp -r /content/project/Result4 /content/drive/MyDrive/compvp_backup/ && echo 'Result4 backed up to My Drive'

In [ ]:
# Step 11: zip Result2/ and Result4/ and save results_colab.zip to the top of My Drive
%cd /content/project
!rm -f results_colab.zip && zip -q -r results_colab.zip Result2 Result4
!cp results_colab.zip /content/drive/MyDrive/ && ls -lh /content/drive/MyDrive/results_colab.zip